In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")

# Display basic information
print("Original Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# Remove unnecessary ID column
df = df.drop(columns=["Car_ID"], errors="ignore")

# Remove duplicate rows
df = df.drop_duplicates()

# Handle missing values
for column in df.columns:
    if df[column].dtype == "object":
        df[column] = df[column].fillna(df[column].mode()[0])
    else:
        df[column] = df[column].fillna(df[column].median())

print("\nDataset Shape After Cleaning:", df.shape)

# Separate features and target
X = df.drop(columns=["Resale_Price_Lakh"])
y = df["Resale_Price_Lakh"]

# Split into training and testing data BEFORE preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

# Identify numerical columns
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical columns
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Treat Condition as ordinal
ordinal_columns = []

if "Condition" in categorical_columns:
    ordinal_columns = ["Condition"]

# Remaining categorical columns are nominal
nominal_columns = [
    column for column in categorical_columns
    if column not in ordinal_columns
]

print("\nNumerical Columns:", numerical_columns)
print("Nominal Columns:", nominal_columns)
print("Ordinal Columns:", ordinal_columns)

# Handle outliers using IQR
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

for column in numerical_columns:
    Q1 = X_train_clean[column].quantile(0.25)
    Q3 = X_train_clean[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    train_outliers = (
        (X_train_clean[column] < lower_bound) |
        (X_train_clean[column] > upper_bound)
    ).sum()

    print(
        f"\n{column}: {train_outliers} outliers "
        f"(Lower = {lower_bound:.2f}, Upper = {upper_bound:.2f})"
    )

    # Clip outliers using training-data boundaries
    X_train_clean[column] = X_train_clean[column].clip(
        lower_bound,
        upper_bound
    )

    X_test_clean[column] = X_test_clean[column].clip(
        lower_bound,
        upper_bound
    )

print("\nOutliers handled using IQR clipping.")

# Create numerical preprocessing pipeline
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# Create nominal encoding pipeline
nominal_pipeline = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# Create ordinal encoding pipeline
transformers = [
    ("numerical", numerical_pipeline, numerical_columns),
    ("nominal", nominal_pipeline, nominal_columns)
]

if len(ordinal_columns) > 0:
    ordinal_pipeline = Pipeline([
        (
            "ordinal",
            OrdinalEncoder(
                categories=[
                    ["Poor", "Fair", "Good", "Very Good", "Excellent"]
                ],
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ])

    transformers.append(
        ("ordinal", ordinal_pipeline, ordinal_columns)
    )

# Create complete preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=transformers
)

# FIT ONLY ON TRAINING DATA
X_train_processed = preprocessor.fit_transform(X_train_clean)

# TRANSFORM TEST DATA WITHOUT FITTING
X_test_processed = preprocessor.transform(X_test_clean)

# Get processed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert processed arrays to DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_clean.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test_clean.index
)

# Add target variable
train_processed = X_train_processed.copy()
train_processed["Resale_Price_Lakh"] = y_train

test_processed = X_test_processed.copy()
test_processed["Resale_Price_Lakh"] = y_test

# Combine processed datasets
processed_dataset = pd.concat(
    [train_processed, test_processed]
).sort_index()

# Verification
print("\n" + "=" * 60)
print("PREPROCESSING VERIFICATION")
print("=" * 60)

print("\nOriginal Dataset Shape:", df.shape)
print("Processed Dataset Shape:", processed_dataset.shape)
print("Training Dataset Shape:", train_processed.shape)
print("Testing Dataset Shape:", test_processed.shape)

print("\nMissing Values in Processed Dataset:")
print(processed_dataset.isnull().sum().sum())

print("\nDuplicate Rows in Processed Dataset:")
print(processed_dataset.duplicated().sum())

print("\nProcessed Dataset Preview:")
print(processed_dataset.head())

print("\nProcessed Data Types:")
print(processed_dataset.dtypes)

# Save datasets
processed_dataset.to_csv(
    "Day12_Used_Car_Preprocessed_Dataset.csv",
    index=False
)

train_processed.to_csv(
    "Day12_Used_Car_Preprocessed_Train.csv",
    index=False
)

test_processed.to_csv(
    "Day12_Used_Car_Preprocessed_Test.csv",
    index=False
)

print("\n" + "=" * 60)
print("FILES SAVED SUCCESSFULLY")
print("=" * 60)

print("\nDay12_Used_Car_Preprocessed_Dataset.csv")
print("Day12_Used_Car_Preprocessed_Train.csv")
print("Day12_Used_Car_Preprocessed_Test.csv")

# Download files automatically in Google Colab
try:
    from google.colab import files

    files.download("Day12_Used_Car_Preprocessed_Dataset.csv")

except:
    print("\nIf using Jupyter Notebook/VS Code, the files are saved in your current folder.")

Original Dataset Shape: (320, 15)

First 5 Rows:
    Car_ID       Brand  Year  Mileage_Km  Engine_CC  Power_BHP Fuel_Type  \
0  CAR0001       Skoda  2021       69708       1152      128.8    Diesel   
1  CAR0002      Toyota  2020       88881        903      146.5    Diesel   
2  CAR0003  Volkswagen  2021       43646       1446      185.9    Diesel   
3  CAR0004        Tata  2019       70847       2069      148.8    Petrol   
4  CAR0005        Tata  2016      101228       1657      206.0    Petrol   

  Transmission        City Seller_Type  Condition  Previous_Owners  \
0       Manual     Lucknow  Individual       Good                1   
1    Automatic  Chandigarh  Individual       Good                1   
2    Automatic   Hyderabad  Individual  Very Good                2   
3       Manual     Lucknow  Individual  Excellent                3   
4    Automatic   Ahmedabad      Dealer  Very Good                2   

   Accidents_Reported  Service_Score  Resale_Price_Lakh  
0              

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>